> ⚠️ **This notebook cannot be run from this repository.**
>
> It shells out to [MAPS](https://github.com/MasonPhonLab/MAPS) (`maps.py`),
> which is **not vendored here**. Only the notebook is kept in this repo, as a
> record of how the MAPS baseline numbers in the paper were produced.
>
> To reproduce:
>
> 1. Clone MAPS separately and set up its own environment. MAPS requires
>    **Python 3.11** and pinned TensorFlow versions, and will conflict with this
>    project's dependencies — do **not** install it here.
>    ```bash
>    git clone https://github.com/MasonPhonLab/MAPS.git
>    conda create -n maps python=3.11 -y
>    conda activate maps
>    cd MAPS && pip install -r requirements.txt
>    ```
> 2. Fix `MAPS_DIR` and the corpus paths (`SRC`, `DST`) in the notebook to match
>    your machine.
> 3. Run this notebook from the MAPS environment.
>
> Notes:
>
> - MAPS does not perform grapheme-to-phoneme conversion and **aborts on any
>   out-of-vocabulary word**, so the notebook builds a CMUdict-derived
>   dictionary and adds pronunciations for the five OOV words in the TIMIT test
>   partition by hand.
> - MAPS does not recurse into subdirectories and writes TextGrids next to the
>   input audio, so the corpus is flattened into a single directory first.
> - The released model (`timbuck_eng.tf`) is trained on the TIMIT training
>   partition and on 35 of the 40 Buckeye speakers, so we evaluate it on the
>   TIMIT test partition only.
> - Numbers reported in the paper were obtained on 2026-08-26; upstream changes
>   to MAPS or to the released model may change the results.

In [ ]:
from pathlib import Path
import shutil, re
from collections import Counter

SRC = Path("/shared/data_zfs/blue2959/timit_test_mfa")
DST = Path("/shared/data_zfs/blue2959/timit_test_maps")

DST.mkdir(parents=True, exist_ok=True)

wavs = sorted(SRC.rglob("*.wav"))
print(f"{len(wavs)} source wavs")

n = 0
for wav in wavs:
    lab = wav.with_suffix(".lab")
    if not lab.exists():
        print("missing lab:", wav)
        continue
    shutil.copy2(wav, DST / wav.name)
    shutil.copy2(lab, (DST / wav.stem).with_suffix(".txt"))
    n += 1

print(f"{n} pairs written to {DST}")
print(f"wav: {len(list(DST.glob('*.wav')))}, txt: {len(list(DST.glob('*.txt')))}")

1680 source wavs
1680 pairs written to /shared/data_zfs/blue2959/timit_test_maps
wav: 1680, txt: 1680


In [2]:
words = Counter()
for txt in sorted(DST.glob("*.txt")):
    for w in txt.read_text().upper().split():
        words[w] += 1

print(f"{len(words)} word types, {sum(words.values())} tokens")
print(list(words.items())[:10])

2378 word types, 14552 tokens
[('SHE', 208), ('HAD', 183), ('YOUR', 202), ('DARK', 171), ('SUIT', 168), ('IN', 313), ('GREASY', 168), ('WASH', 168), ('WATER', 170), ('ALL', 223)]


In [3]:
DEMO_DICT = Path.home() / "MAPS/MAPS/demo_files/sample_dictionary.txt"
print(DEMO_DICT.read_text().splitlines()[:10])

['ALL  AA1 L', 'DARK  D AA1 R K', 'GREASY  G R IY1 S IY0', 'HAD  HH AE1 D', 'IN  IH1 N', 'SHE  SH IY1', 'SUIT  S UW1 T', 'WASH  W AA1 SH', 'WATER  W AA1 T ER0', 'YEAR  Y IH1 R']


In [4]:
UTILS = Path.home() / "MAPS/MAPS/utils.py"
src = UTILS.read_text()
i = src.find("def load_dictionary")
print(src[i:i+1200])

def load_dictionary(dname, cmuformat=True):

    

    mapping = dict()
    
    if cmuformat:

        with open(dname, 'r') as d:
            for line in d:
                all_items = line.split()
                word = all_items[0]
                word = re.sub(r'\(\d*\)', '', word)
                pronunciation = all_items[1:]
                pronunciation = [fold_phone(p) for p in pronunciation]
                
                if word not in mapping:
                    mapping[word] = [pronunciation]
                else:
                    mapping[word].append(pronunciation)
                    
        mapping['sil'] = [['H#'], []]
        return mapping
        
    else:
        print('Custom dictionary formats not supported yet. Please format use CMU dict formatting and run again.')
        sys.exit(1)
                



In [5]:
import urllib.request

CMU = DST.parent / "cmudict.dict"
if not CMU.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/cmusphinx/cmudict/master/cmudict.dict",
        CMU)
print(CMU, CMU.stat().st_size)
print(CMU.read_text().splitlines()[:5])

cmu_words = set()
for line in CMU.read_text().splitlines():
    if not line.strip() or line.startswith(";;;"):
        continue
    cmu_words.add(line.split()[0].upper())

oov = {w: c for w, c in words.items() if w not in cmu_words}
print(f"\n{len(oov)} OOV types, {sum(oov.values())} tokens")
for w, c in sorted(oov.items(), key=lambda x: -x[1]):
    print(f"{c:5d}  {w}")

/shared/data_zfs/blue2959/cmudict.dict 3618488
["'bout B AW1 T", "'cause K AH0 Z", "'course K AO1 R S", "'cuse K Y UW1 Z", "'em AH0 M"]

5 OOV types, 5 tokens
    1  EXHUSBAND
    1  MOTORISTS'
    1  NIHILISTIC
    1  RADIOSTERILIZATION
    1  MORPHOPHONEMIC


In [6]:
import re

OOV_PRONS = {
    "EXHUSBAND":          "EH1 K S HH AH1 Z B AH0 N D",
    "MOTORISTS'":         "M OW1 T ER0 IH0 S T S",
    "NIHILISTIC":         "N AY2 AH0 L IH1 S T IH0 K",
    "RADIOSTERILIZATION": "R EY2 D IY0 OW0 S T EH2 R AH0 L AH0 Z EY1 SH AH0 N",
    "MORPHOPHONEMIC":     "M AO2 R F OW0 F OW0 N IY1 M IH0 K",
}

MAPS_DICT = DST.parent / "cmudict_maps.txt"

seen = set()
lines = []
for line in CMU.read_text().splitlines():
    line = line.strip()
    if not line or line.startswith(";;;"):
        continue
    parts = line.split()
    w = re.sub(r"\(\d+\)", "", parts[0]).upper()
    if w in seen:            # 변이형 제거, 첫 항목만 유지
        continue
    seen.add(w)
    # 주석(#) 이후 제거
    prons = []
    for p in parts[1:]:
        if p.startswith("#"):
            break
        prons.append(p.upper())
    if not prons:
        continue
    lines.append(f"{w}  {' '.join(prons)}")

for w, p in OOV_PRONS.items():
    if w not in seen:
        lines.append(f"{w}  {p}")
        seen.add(w)

MAPS_DICT.write_text("\n".join(lines) + "\n")
print(f"{len(lines)} entries -> {MAPS_DICT}")

# 검증
missing = [w for w in words if w not in seen]
print(f"remaining OOV: {len(missing)}  {missing}")

126057 entries -> /shared/data_zfs/blue2959/cmudict_maps.txt
remaining OOV: 0  []


In [7]:
import subprocess

MAPS_DIR = Path.home() / "MAPS/MAPS"
cmd = [
    "python", "maps.py",
    f"--audio={DST}",
    f"--text={DST}",
    f"--dict={MAPS_DICT}",
    "--model=timbuck_eng.tf",
    "--overwrite",
]
print(" ".join(cmd))

p = subprocess.run(cmd, cwd=MAPS_DIR, capture_output=True, text=True)
print(p.stdout[-3000:])
print("STDERR:", p.stderr[-3000:])

python maps.py --audio=/shared/data_zfs/blue2959/timit_test_maps --text=/shared/data_zfs/blue2959/timit_test_maps --dict=/shared/data_zfs/blue2959/cmudict_maps.txt --model=timbuck_eng.tf --overwrite
BEGINNING ALIGNMENT
USING MODEL timbuck_eng.tf (1/1)

STDERR:  1613/1680 [03:21<00:08,  7.95it/s]
100%|██████████| 1680/1680 [03:29<00:00,  8.02it/s]



In [ ]:
tgs = sorted(DST.glob("*.TextGrid"))
print(len(tgs))
print(tgs[0].read_text()[:500])

1680
File type = "ooTextFile"
Object class = "TextGrid"

xmin = 0.0
xmax = 3.44325
tiers? <exists>
size = 2
item []:
	item [1]:
		class = "IntervalTier"
		name = "words"
		xmin = 0.0
		xmax = 3.44325
		intervals: size = 13
			intervals [1]:
				xmin = 0
				xmax = 0.17868706645643673
				text = "sil"
			intervals [2]:
				xmin = 0.17868706645643673
				xmax = 0.3032929958746532
				text = "SHE"
			intervals [3]:
				xmin = 0.3032929958746532
				xmax = 0.5577340490101355
				text = "HAD"
			intervals


In [9]:
import numpy as np
import textgrid


def read_word_intervals(tg_path):
    tg = textgrid.TextGrid.fromFile(str(tg_path))
    tier = next(t for t in tg.tiers if t.name.lower() == "words")
    return [(iv.minTime, iv.maxTime, iv.mark.strip().lower())
            for iv in tier if iv.mark.strip()]


def read_ref(wrd_path, sr=16000):
    ref = []
    for line in open(wrd_path):
        line = line.strip()
        if not line:
            continue
        s, e, w = line.split(maxsplit=2)
        ref.append((int(s) / sr, int(e) / sr, w.strip().lower()))
    return ref


def lcs_match(ref_words, hyp_words):
    n, m = len(ref_words), len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n):
        for j in range(m):
            if ref_words[i] == hyp_words[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])
    pairs = []
    i, j = n, m
    while i > 0 and j > 0:
        if ref_words[i - 1] == hyp_words[j - 1]:
            pairs.append((i - 1, j - 1))
            i -= 1
            j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    pairs.reverse()
    return pairs


def report(name, errs):
    e = np.array(errs)
    print(f"{name:12s}  n={len(e):7d}  WBE={e.mean()*1000:6.2f} ms  "
          + f"P10={100*(e<=.010).mean():5.2f}  P25={100*(e<=.025).mean():5.2f}  "
          + f"P50={100*(e<=.050).mean():5.2f}  P100={100*(e<=.100).mean():5.2f}")

In [10]:
WRD_ROOT = Path("/shared/data_zfs/blue2959/TIMIT/TEST")
wrd_index = {(p.parent.name, p.stem): p for p in WRD_ROOT.rglob("*.WRD")}

all_errs, all_starts, all_ends = [], [], []
n_missing = 0
total_bounds = kept_bounds = 0

for tg in sorted(DST.glob("*.TextGrid")):
    spk, utt = tg.stem.split("_", 1)
    wrd = wrd_index.get((spk, utt))
    if wrd is None:
        n_missing += 1
        continue

    ref = read_ref(wrd)
    total_bounds += 2 * len(ref)

    hyp = [x for x in read_word_intervals(tg) if x[2] != "sil"]
    pairs = lcs_match([w for *_, w in ref], [w for *_, w in hyp])

    for ri, hi in pairs:
        rs, re_, _ = ref[ri]
        hs, he, _ = hyp[hi]
        all_starts.append(abs(hs - rs))
        all_ends.append(abs(he - re_))
        all_errs.extend([abs(hs - rs), abs(he - re_)])
        kept_bounds += 2

print(f"missing={n_missing}")
print(f"boundary coverage = {100*kept_bounds/total_bounds:.2f}%  "
      + f"({kept_bounds}/{total_bounds})\n")
report("start+end", all_errs)
report("start only", all_starts)
report("end only", all_ends)

missing=0
boundary coverage = 99.99%  (29104/29106)

start+end     n=  29104  WBE= 21.38 ms  P10=49.17  P25=76.75  P50=89.92  P100=96.95
start only    n=  14552  WBE= 20.93 ms  P10=50.89  P25=78.06  P50=90.25  P100=97.07
end only      n=  14552  WBE= 21.82 ms  P10=47.46  P25=75.43  P50=89.59  P100=96.83
